# Cross-validation for time series (the [`sktime`](https://www.sktime.net/en/stable/) package)

We will consider this using an ARIMA model as an example

In [ ]:
import numpy as np
import pandas as pd

# Forecasting model, e.g. ARIMA
from sktime.forecasting.arima import ARIMA
# Time series visualization
from sktime.utils.plotting import plot_series
# Modules for cross-validation
from sktime.split import temporal_train_test_split, ExpandingWindowSplitter, SlidingWindowSplitter, SingleWindowSplitter
from sktime.forecasting.model_evaluation import evaluate
from sktime.performance_metrics.forecasting import MeanSquaredError, MeanAbsoluteError, MeanAbsolutePercentageError # Metrics MSE, MAE, MAPE

import pandas_datareader.data as web

# visualization settings
import matplotlib.pyplot as plt

# Do not show Warnings
import warnings
warnings.simplefilter(action='ignore', category=Warning)
# Do not show ValueWarning, ConvergenceWarning from statsmodels
from statsmodels.tools.sm_exceptions import ValueWarning, ConvergenceWarning
warnings.simplefilter('ignore', category=ValueWarning)
warnings.simplefilter('ignore', category=ConvergenceWarning)

Load monthly data from the [`FRED`](https://fred.stlouisfed.org/) database on Market Yield on U.S. Treasury Securities at 10-Year Constant Maturity (Symbol [`GS10`](https://fred.stlouisfed.org/series/GS10)) from 2000-01-01 to 2023-12-31 and create a dataframe `y`

In [ ]:
y = web.DataReader(name='GS10', data_source='fred', start='2000-01', end='2023-12')
y.index = y.index.to_period(freq='M')
# series length
len(y)

## Validation via train/test split

Use the [`temporal_train_test_split`](https://www.sktime.net/en/stable/api_reference/auto_generated/sktime.split.temporal_train_test_split.html) method from the `sktime` package

Split the series into a training set (the first, for example 80%, of observations) and a test set (the last, in our case 20%, of observations)

As the metric we will choose [`MeanSquaredError`](https://www.sktime.net/en/stable/api_reference/auto_generated/sktime.performance_metrics.forecasting.MeanSquaredError.html)

We will consider this using ARIMA(2,1,2) as an example

In [ ]:
# split the sample into training and test (the last 10 observations)
y_train, y_test = temporal_train_test_split(y, test_size=10)

# initialize the metric
metric = MeanSquaredError(square_root=False)

In [ ]:
# specify the forecasting model, e.g. ARIMA(2,1,2)
forecaster = ARIMA(order=(2,1,2), trend='n')

# Fit the model on the training set and forecast for the test observations
y_pred = forecaster.fit_predict(y_train, fh = y_test.index)

# compute the metric on the data
metric.evaluate(y_test, y_pred)

In [ ]:
plot_series(y_train.tail(30), y_test, y_pred, labels=['train', 'test', 'pred'])

plt.show()

Use [`SingleWindowSplitter`](https://www.sktime.net/en/stable/api_reference/auto_generated/sktime.split.SingleWindowSplitter.html) 

To compute the metric for each fold, use the [`MeanSquaredError`](https://www.sktime.net/en/stable/api_reference/auto_generated/sktime.performance_metrics.forecasting.MeanSquaredError.html), [`MeanAbsoluteError`](https://www.sktime.net/en/stable/api_reference/auto_generated/sktime.performance_metrics.forecasting.MeanAbsoluteError.html), [`MeanAbsolutePercentageError`](https://www.sktime.net/en/stable/api_reference/auto_generated/sktime.performance_metrics.forecasting.MeanAbsolutePercentageError.html#sktime.performance_metrics.forecasting.MeanAbsolutePercentageError) methods

To perform cross-validation, use [`evaluate`](https://www.sktime.net/en/stable/api_reference/auto_generated/sktime.forecasting.model_evaluation.evaluate.html)

We will consider this using ARMA(2,1,2) without drift as an example

In [ ]:
# specify the forecasting model, e.g. ARIMA(2,1,2) without drift
forecaster = ARIMA(order=(2,1,2), trend='n')

# split the sample into training (the window_length parameter) and test (the fh parameter)
cv_strategy = SingleWindowSplitter(fh=np.arange(1, 6), window_length=len(y)-5)

# initialize the metrics
metric = [MeanSquaredError(square_root=False), MeanAbsoluteError(), MeanAbsolutePercentageError()]

cv_res = evaluate(forecaster=forecaster, y=y, cv=cv_strategy, strategy="refit", return_data=False, scoring=metric)
cv_res

In [ ]:
# MSE, MAE, MAPE
cv_res.iloc[:,:len(metric)]

## Validation using the k-Fold method (expanding training set)

Use [`ExpandingWindowSplitter`](https://www.sktime.net/en/stable/api_reference/auto_generated/sktime.split.ExpandingWindowSplitter.html) 

To compute the metric for each fold, use the [`MeanSquaredError`](https://www.sktime.net/en/stable/api_reference/auto_generated/sktime.performance_metrics.forecasting.MeanSquaredError.html), [`MeanAbsoluteError`](https://www.sktime.net/en/stable/api_reference/auto_generated/sktime.performance_metrics.forecasting.MeanAbsoluteError.html), [`MeanAbsolutePercentageError`](https://www.sktime.net/en/stable/api_reference/auto_generated/sktime.performance_metrics.forecasting.MeanAbsolutePercentageError.html#sktime.performance_metrics.forecasting.MeanAbsolutePercentageError) methods

To perform cross-validation, use [`evaluate`](https://www.sktime.net/en/stable/api_reference/auto_generated/sktime.forecasting.model_evaluation.evaluate.html)

We will consider this using ARIMA(2,1,2) without drift as an example

In [ ]:
# specify the forecasting model, e.g. ARIMA(2,1,2) without drift
forecaster = ARIMA(order=(2,1,2), trend='n')

# split the sample into training (starting with the first initial_window) and test of length (the fh parameter)
# then expand the training set by step_length observations
cv_strategy = ExpandingWindowSplitter(fh=np.arange(1, 11), initial_window=100, step_length=10)

# initialize the metrics
metric = [MeanSquaredError(square_root=False), MeanAbsoluteError(), MeanAbsolutePercentageError()]

cv_res = evaluate(forecaster=forecaster, y=y, cv=cv_strategy, strategy="refit", return_data=False, scoring=metric)
cv_res

In [ ]:
# mean MSE, MAE, MAPE
cv_res.iloc[:,:len(metric)].mean()

## Validation using the k-Fold method (sliding training set)

Use the [`SlidingWindowSplitter`](https://www.sktime.net/en/stable/api_reference/auto_generated/sktime.split.SlidingWindowSplitter.html) method

To compute the metric for each fold, use the [`MeanSquaredError`](https://www.sktime.net/en/stable/api_reference/auto_generated/sktime.performance_metrics.forecasting.MeanSquaredError.html), [`MeanAbsoluteError`](https://www.sktime.net/en/stable/api_reference/auto_generated/sktime.performance_metrics.forecasting.MeanAbsoluteError.html), [`MeanAbsolutePercentageError`](https://www.sktime.net/en/stable/api_reference/auto_generated/sktime.performance_metrics.forecasting.MeanAbsolutePercentageError.html#sktime.performance_metrics.forecasting.MeanAbsolutePercentageError) methods

To perform cross-validation, use [`evaluate`](https://www.sktime.net/en/stable/api_reference/auto_generated/sktime.forecasting.model_evaluation.evaluate.html)

We will consider this using ARIMA(2,1,2) without drift as an example

In [ ]:
# specify the forecasting model, e.g. ARIMA(2,1,2) without drift
forecaster = ARIMA(order=(2,1,2), trend='n')

# split the sample into training (of length initial_window) and test (the fh parameter)
# then shift the training and test sets by step_length steps
cv_strategy = SlidingWindowSplitter(fh=np.arange(1, 11), initial_window=100, step_length=10)

# initialize the metrics
metric = [MeanSquaredError(square_root=False), MeanAbsoluteError(), MeanAbsolutePercentageError()]

cv_res = evaluate(forecaster=forecaster, y=y, cv=cv_strategy, strategy="refit", return_data=False, scoring=metric)
cv_res

In [ ]:
# mean MSE, MAE, MAPE
cv_res.iloc[:,:len(metric)].mean()

## Cross-validation for several models

Perform cross-validation to compare several models. For example, compare
* ARIMA(1,1,1) without drift
* ARIMA(1,1,1) with drift
* ARIMA(1,2,1) without drift

In [ ]:
# Define a list of specified forecasting models
forecasters = [ARIMA(order=(1,1,1), trend='n'), ARIMA(order=(1,1,1), trend='c'), ARIMA(order=(1,2,1), trend='n')]

# specify the cross-validation method. For example, SlidingWindowSplitter
cv_strategy = SlidingWindowSplitter(fh=np.arange(1, 6), initial_window=100, step_length=5)

# initialize the metrics
metric = [MeanSquaredError(square_root=False), MeanAbsoluteError(), MeanAbsolutePercentageError()]

# dataframe with metrics by columns
cv_data = pd.DataFrame(data=None, columns=['MSE', 'MAE', 'MAPE'])

for model in forecasters:
	print(model)
	cv_res = evaluate(forecaster=model, y=y, cv=cv_strategy, strategy="refit", return_data=False, scoring=metric)
	# print(df.iloc[:,:len(metric)].mean()) # metrics for each model
	cv_data.loc[len(cv_data.index)] = cv_res.iloc[:,[0,1,2]].mean().values

# cross-validation results as a dataframe
cv_data

In [ ]:
# Model index
for i in range(cv_data.shape[1]):
	print(f'{cv_data.columns[i]}: model #={cv_data.iloc[:,i].argmin()}')